In [0]:
dbutils.widgets.removeAll()

In [0]:
# Databricks notebook source
# DBTITLE 1,Setup logging
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
base_dir = os.getcwd()
log_dir = os.path.abspath(os.path.join(base_dir, "../../logs"))
os.makedirs(log_dir, exist_ok=True)

# Create log filename with timestamp
log_filename = f"{log_dir}/member_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Member silver Layer Processing Started")
logger.info(f"Log file: {log_filename}")
logger.info("=" * 60)

In [0]:
#n  Databricks notebook source
# DBTITLE 1,Configuration - Set paths and parameters

# Source paths - updated to use actual Unity Catalog table locations
sourcePath = "/Volumes/claimsprocessing/bronze/member_consolidated"
silverMemPerBrdgTable = "claimsprocessing.silver.silver_member_person_bridge"
silverMemTable = "claimsprocessing.silver.silver_member"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Silver Person Bridge Table: {silverMemPerBrdgTable}")
print(f"Silver Member Table: {silverMemTable}")
print("="*60)

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Person Bridge Table: {silverMemPerBrdgTable}")
logger.info(f"  Silver Member Table: {silverMemTable}")

In [0]:
# Databricks notebook source
# DBTITLE 1,Helper functions

def table_exists(tableToCheck):
    """Check if a Unity Catalog table exists"""
    try:
        spark.table(tableToCheck)
        logger.info(f"Table check: {tableToCheck} - Table exists")
        return True
    except Exception as e:
        logger.warning(f"Table check: {tableToCheck} - Table does not exist or is inaccessible: {str(e)}")
        return False

In [0]:
# Databricks notebook source
# DBTITLE 1,Import libraries

from pyspark.sql.functions import date_format, monotonically_increasing_id, sha2, concat_ws, col, row_number
from pyspark.sql.window import Window

In [0]:
# Databricks notebook source
# DBTITLE 1,Define SQL transformations

# SQL to join consolidated member data with person bridge
srcsql = """
WITH consolidateMem1 as (
  SELECT 
    brdg.ESAIInternalPersonID, brdg.UniqueRecord, mem.ClientID, mem.FileID, mem.LoadDateTime, 
    mem.FileLayoutID, mem.FileLayoutDescription, mem.UniquePersonKey, mem.PlanMemberID, 
    mem.SubscriberID, mem.BeneficiaryID, mem.LastName, mem.FirstName, mem.MiddleInitial, 
    mem.EnrolleeUniqueID, mem.DateofBirth, mem.DeceasedDate, mem.Gender, 
    mem.PermanentAddressLine1, mem.PermanentAddressLine2, mem.PermanentCity, 
    mem.PermanentCounty, mem.PermanentState, mem.PermanentZipCode, 
    mem.MailingAddressLine1, mem.MailingAddressLine2, mem.MailingCity, 
    mem.MailingState, mem.MailingZipCode, mem.MailingCounty, mem.PhoneNumber, 
    mem.Email, mem.MedicaidID, mem.Fax, mem.RaceCode, mem.RaceDataSource, 
    mem.CaretakerFirstName, mem.CaretakerLastName, mem.CaretakerMiddleInitial, 
    mem.EthnicityCode, mem.EthnicityDatasource, mem.SpokenLanguage, 
    mem.SpokenLanguagesourcecode, mem.WrittenLanguageCode, mem.WrittenLanguageSourcecode, 
    mem.OtherLanguage, mem.OtherLanguageSourcecode, mem.USCitizen, 
    mem.AlternateKey1, mem.AlternateKey2, mem.AlternateKey3, mem.AlternateKey4, 
    mem.AlternateKey5, mem.AlternateKey6, mem.AlternateKey7, mem.AlternateKey8, 
    mem.AlternateKey9, mem.AlternateKey10, mem.MaskedMemberID, mem.EnrolleeEducation, 
    mem.EnrolleeEmployment, brdg.PMUP, brdg.IsCurrentPMUP, mem.ProductID
  FROM consolidateMem mem
  INNER JOIN silverMemPerBrdg brdg 
    ON mem.UniqueRecord = brdg.UniqueRecord 
    AND mem.FileLayoutID = brdg.FileLayoutID 
    AND brdg.IsCurrentPMUP = 1
)
SELECT 
  ESAIInternalPersonID, UniqueRecord, ClientID, FileID, LoadDateTime, FileLayoutID, 
  FileLayoutDescription, UniquePersonKey, PlanMemberID, SubscriberID, BeneficiaryID, 
  LastName, FirstName, MiddleInitial, EnrolleeUniqueID, DateofBirth, DeceasedDate, 
  Gender, PermanentAddressLine1, PermanentAddressLine2, PermanentCity, PermanentCounty, 
  PermanentState, PermanentZipCode, MailingAddressLine1, MailingAddressLine2, 
  MailingCity, MailingState, MailingZipCode, MailingCounty, PhoneNumber, Email, 
  MedicaidID, Fax, RaceCode, RaceDataSource, CaretakerFirstName, CaretakerLastName, 
  CaretakerMiddleInitial, EthnicityCode, EthnicityDatasource, SpokenLanguage, 
  SpokenLanguagesourcecode, WrittenLanguageCode, WrittenLanguageSourcecode, 
  OtherLanguage, OtherLanguageSourcecode, USCitizen, AlternateKey1, AlternateKey2, 
  AlternateKey3, AlternateKey4, AlternateKey5, AlternateKey6, AlternateKey7, 
  AlternateKey8, AlternateKey9, AlternateKey10, MaskedMemberID, EnrolleeEducation, 
  EnrolleeEmployment, PMUP, IsCurrentPMUP, ProductID,
  sha2(concat(
    IfNull(ESAIInternalPersonID,""), "|", IfNull(UniqueRecord,""), "|", 
    IfNull(ClientID,""), "|", IfNull(CAST(FileID AS STRING),""), "|", 
    IfNull(CAST(LoadDateTime AS STRING),""), "|", IfNull(CAST(FileLayoutID AS STRING),""), "|", 
    IfNull(FileLayoutDescription,""), "|", IfNull(UniquePersonKey,""), "|", 
    IfNull(PlanMemberID,""), "|", IfNull(SubscriberID,""), "|", 
    IfNull(BeneficiaryID,""), "|", IfNull(LastName,""), "|", IfNull(FirstName,""), "|", 
    IfNull(MiddleInitial,""), "|", IfNull(EnrolleeUniqueID,""), "|", 
    IfNull(CAST(DateofBirth AS STRING),""), "|", IfNull(CAST(DeceasedDate AS STRING),""), "|", 
    IfNull(Gender,""), "|", IfNull(PermanentAddressLine1,""), "|", 
    IfNull(PermanentAddressLine2,""), "|", IfNull(PermanentCity,""), "|", 
    IfNull(PermanentCounty,""), "|", IfNull(PermanentState,""), "|", 
    IfNull(PermanentZipCode,""), "|", IfNull(MailingAddressLine1,""), "|", 
    IfNull(MailingAddressLine2,""), "|", IfNull(MailingCity,""), "|", 
    IfNull(MailingState,""), "|", IfNull(MailingZipCode,""), "|", 
    IfNull(MailingCounty,""), "|", IfNull(PhoneNumber,""), "|", IfNull(Email,""), "|", 
    IfNull(MedicaidID,""), "|", IfNull(Fax,""), "|", IfNull(RaceCode,""), "|", 
    IfNull(RaceDataSource,""), "|", IfNull(CaretakerFirstName,""), "|", 
    IfNull(CaretakerLastName,""), "|", IfNull(CaretakerMiddleInitial,""), "|", 
    IfNull(EthnicityCode,""), "|", IfNull(EthnicityDatasource,""), "|", 
    IfNull(SpokenLanguage,""), "|", IfNull(SpokenLanguagesourcecode,""), "|", 
    IfNull(WrittenLanguageCode,""), "|", IfNull(WrittenLanguageSourcecode,""), "|", 
    IfNull(OtherLanguage,""), "|", IfNull(OtherLanguageSourcecode,""), "|", 
    IfNull(USCitizen,""), "|", IfNull(AlternateKey1,""), "|", IfNull(AlternateKey2,""), "|", 
    IfNull(AlternateKey3,""), "|", IfNull(AlternateKey4,""), "|", 
    IfNull(AlternateKey5,""), "|", IfNull(AlternateKey6,""), "|", 
    IfNull(AlternateKey7,""), "|", IfNull(AlternateKey8,""), "|", 
    IfNull(AlternateKey9,""), "|", IfNull(AlternateKey10,""), "|", 
    IfNull(MaskedMemberID,""), "|", IfNull(EnrolleeEducation,""), "|", 
    IfNull(EnrolleeEmployment,""), "|", IfNull(PMUP,""), "|", 
    IfNull(CAST(IsCurrentPMUP AS STRING),""), "|", IfNull(ProductID,"")
  ), 256) AS HashKey 
FROM consolidateMem1
"""

In [0]:
# Databricks notebook source
# DBTITLE 1,Main execution - Build Silver Member table

logger.info("=" * 60)
logger.info("MAIN EXECUTION STARTED")
logger.info("=" * 60)
print(f"\nProcessing Silver Member data...")
logger.info("Processing Silver Member data...")
print(f"Source: {sourcePath}")
print(f"Person Bridge: {silverMemPerBrdgTable}")
logger.info(f"Source: {sourcePath}")
logger.info(f"Person Bridge: {silverMemPerBrdgTable}")

logger.info(f"Checking if source table exists: {sourcePath}")
if table_exists(sourcePath):
    logger.info("Source table exists - proceeding with processing")
    # Load consolidated member data
    print("Loading consolidated member data...")
    logger.info("Loading consolidated member data from source table...")
    dfconsolidateMemSrc = spark.read.format("delta").load(sourcePath)
    source_count = dfconsolidateMemSrc.count()
    logger.info(f"Consolidated member data loaded: {source_count} records")
    
    # Load person bridge table
    print("Loading person bridge data...")
    logger.info(f"Loading person bridge table: {silverMemPerBrdgTable}")
    dfsilverMemPerBrdg = spark.table(silverMemPerBrdgTable)
    bridge_count = dfsilverMemPerBrdg.count()
    logger.info(f"Person bridge data loaded: {bridge_count} records")
    
    # Add UniqueRecord column to consolidated data
    logger.info("Starting data transformations...")
    windowPartition = Window.partitionBy(col("FileId")).orderBy(col("RecordHash").desc())

    # Step 1: Remove completely identical rows
    # Step 2: Create a temporary fingerprint of the raw source data
    # Step 3: Give every row a sequence number (1, 2, 3...) inside its FileId bucket
    # Step 4: Glue FileID and RowNumber together to create a permanent system ID (e.g., "01-1")
    logger.info("Applying transformations: distinct, hash, row number, unique record")

    dfconsolidateMem = dfconsolidateMemSrc.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfconsolidateMemSrc.columns), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FileID"), col("RowNumber")))
    
    transformed_count = dfconsolidateMem.count()
    logger.info(f"Transformations completed: {transformed_count} records after distinct")
    
    # Create temp views
    logger.info("Creating temporary views for SQL processing")
    dfconsolidateMem.createOrReplaceTempView("consolidateMem")
    logger.info("Created temp view: consolidateMem")
    dfsilverMemPerBrdg.createOrReplaceTempView("silverMemPerBrdg")
    logger.info("Created temp view: silverMemPerBrdg")
    
    # Join member data with person bridge
    print("Joining member data with person bridge...")
    logger.info("Executing SQL join between member and person bridge...")
    dfsrc = spark.sql(srcsql)
    logger.info("SQL join completed")
    
    recordCount = dfsrc.count()
    print(f"Found {recordCount} records to process")
    logger.info(f"Join result: {recordCount} records to process")
    
    if recordCount > 0:
        # Write to silver table
        print(f"\nWriting to Silver Member table: {silverMemTable}")
        logger.info(f"Writing {recordCount} records to Silver Member table: {silverMemTable}")
        dfsrc.write.format("delta").mode("append").saveAsTable(silverMemTable)
        print("\n Silver Member table created successfully!")
        logger.info("Silver Member table created successfully")
        logger.info("=" * 60)
        logger.info("PROCESSING COMPLETED SUCCESSFULLY")
        logger.info("=" * 60)
    else:
        print("\n No records to process - check person bridge table has data")
        logger.warning("No records to process - join returned 0 records")
        logger.warning("Check if person bridge table has data with IsCurrentPMUP = 1")
        logger.info("=" * 60)
        logger.info("PROCESSING COMPLETED - NO RECORDS")
        logger.info("=" * 60)
else:
    print(f"\n Source table does not exist: {sourcePath}")
    logger.error(f"Source table does not exist: {sourcePath}")
    logger.info("=" * 60)
    logger.info("PROCESSING ABORTED - SOURCE TABLE NOT FOUND")
    logger.info("=" * 60)